# 🔴 CascadeGuard — SFT + GRPO Training (Colab T4, Pure TRL)

**Meta × PyTorch OpenEnv Hackathon — Submission notebook.**

### What this notebook does
1. Verifies T4 GPU
2. Installs **TRL + PEFT + BitsAndBytes** (no Unsloth, no MergeKit)
3. Clones & installs `cascade_gaurd_openEnv`
4. Pre-warms the OSM environment
5. **SFT warm-start** from the pre-built `cascadeguard_sft_dataset.csv` — teaches all action formats
6. **GRPO training** with live `CascadeEnvironment` reward signal
7. Plots loss + reward curves, runs baseline comparison
8. Commits evidence to GitHub

> **Before starting:** `Runtime → Change runtime type → T4 GPU`

## ✅ Step 0 — Verify GPU

In [1]:
import subprocess, sys, torch

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout[:600])

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU     : {p.name}")
    print(f"VRAM    : {p.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("⛔ No GPU. Runtime → Change runtime type → T4 GPU")

Sat Apr 25 14:44:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|===============
PyTorch : 2.10.0+cu128
CUDA    : 12.8
GPU     : Tesla T4
VRAM    : 15.6 GB


## 📦 Step 1 — Install all dependencies

Pure TRL stack — no Unsloth, no MergeKit.

In [2]:
import subprocess, sys

def pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet', *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
    return r.returncode

print("Installing TRL / PEFT / Accelerate / BitsAndBytes ...")
pip('trl>=0.12.0', 'datasets>=4.2', 'peft>=0.14',
    'accelerate>=1.10', 'bitsandbytes', 'transformers>=4.40')

print("Installing osmnx + geospatial stack ...")
pip('osmnx>=1.9.3', 'shapely>=2.0', 'geopandas', 'networkx>=3.0')

print("Installing project utilities ...")
pip('pandas', 'matplotlib', 'seaborn',
    'fastapi>=0.115', 'uvicorn>=0.30', 'pydantic>=2.0',
    'openenv-core[core]>=0.2.2', 'httpx')

print("\n✅ All packages installed.")

Installing TRL / PEFT / Accelerate / BitsAndBytes ...
Installing osmnx + geospatial stack ...
Installing project utilities ...

✅ All packages installed.


## 🐙 Step 2 — Clone repo & detect package layout

In [3]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Samarth-Dave/cascade_gaurd_openEnv'
REPO_DIR = Path('/content/cascade_gaurd_openEnv')

if REPO_DIR.is_dir():
    print("Repo already present — pulling ...")
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

PKG_AS_SUBDIR = REPO_DIR
PKG_AS_ROOT   = REPO_DIR / 'server'

if (PKG_AS_SUBDIR / 'server' / 'cascade_environment.py').exists():
    PACKAGE_ROOT = REPO_DIR
    print(f"✅ Layout: cascade_guard/ sub-package under {REPO_DIR.name}/")
elif (PKG_AS_ROOT / 'cascade_environment.py').exists():
    PACKAGE_ROOT = REPO_DIR
    print("✅ Layout: server/ at repo root (flat layout)")
else:
    print("⚠️  Could not detect layout — listing repo root:")
    for p in sorted(REPO_DIR.iterdir()):
        print(f"   {p.name}{'/' if p.is_dir() else ''}")
    PACKAGE_ROOT = REPO_DIR

print(f"\nREPO_DIR     : {REPO_DIR}")
print(f"PACKAGE_ROOT : {PACKAGE_ROOT}")

✅ Layout: cascade_guard/ sub-package under cascade_gaurd_openEnv/

REPO_DIR     : /content/cascade_gaurd_openEnv
PACKAGE_ROOT : /content/cascade_gaurd_openEnv


## 📂 Step 3 — Install `cascade_guard` package & verify imports

In [4]:
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/cascade_gaurd_openEnv').resolve()
ALIAS_DIR = Path('/content/cascade_guard')

assert REPO_DIR.is_dir(), f'Repo dir missing: {REPO_DIR}. Re-run Step 2 to clone.'
assert (REPO_DIR / 'server' / 'cascade_environment.py').exists(), (
    f'Repo at {REPO_DIR} does not contain server/cascade_environment.py — wrong branch?'
)

# Drop any partial cascade_guard module objects from earlier failed cell runs.
for mod_name in [m for m in list(sys.modules) if m == 'cascade_guard' or m.startswith('cascade_guard.')]:
    del sys.modules[mod_name]

# Strategy 1: try pip editable install (works only if the GitHub branch
# carries the package-dir mapping in pyproject.toml).
pip_proc = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if pip_proc.returncode != 0:
    print('pip install -e returned non-zero — falling back to symlink alias.')
    print((pip_proc.stderr or '')[-1500:])

for path_entry in (str(REPO_DIR.parent), str(REPO_DIR)):
    if path_entry not in sys.path:
        sys.path.insert(0, path_entry)
importlib.invalidate_caches()

import_strategy = None
try:
    import cascade_guard  # noqa: F401
    import_strategy = 'pip-editable-or-syspath'
except ModuleNotFoundError:
    # Strategy 2: symlink /content/cascade_guard -> /content/cascade_gaurd_openEnv
    # so that "import cascade_guard" resolves regardless of pyproject layout.
    if ALIAS_DIR.exists() or ALIAS_DIR.is_symlink():
        try:
            if ALIAS_DIR.is_symlink() or ALIAS_DIR.is_file():
                ALIAS_DIR.unlink()
            else:
                shutil.rmtree(ALIAS_DIR)
        except Exception as exc:
            print('Could not remove existing alias dir:', exc)

    try:
        os.symlink(str(REPO_DIR), str(ALIAS_DIR), target_is_directory=True)
        print(f'Created symlink alias: {ALIAS_DIR} -> {REPO_DIR}')
    except OSError:
        shutil.copytree(REPO_DIR, ALIAS_DIR, symlinks=False, ignore_dangling_symlinks=True)
        print(f'Copied repo into alias dir: {ALIAS_DIR}')

    init_file = ALIAS_DIR / '__init__.py'
    if not init_file.exists():
        init_file.write_text('"""cascade_guard package shim (Colab alias)."""\n', encoding='utf-8')

    if str(ALIAS_DIR.parent) not in sys.path:
        sys.path.insert(0, str(ALIAS_DIR.parent))

    for mod_name in [m for m in list(sys.modules) if m == 'cascade_guard' or m.startswith('cascade_guard.')]:
        del sys.modules[mod_name]
    importlib.invalidate_caches()

    import cascade_guard  # noqa: F401
    import_strategy = 'symlink-alias'

from cascade_guard.server.cascade_environment import CascadeEnvironment
from cascade_guard.models import CascadeAction, CascadeObservation
from cascade_guard.tasks import TASK_CONFIGS

print('cascade_guard package path             :', cascade_guard.__file__)
print('Import strategy used                  :', import_strategy)
print('from cascade_guard.models             : ✓')
print('from cascade_guard.tasks              : ✓')
print('from cascade_guard.server.cascade_env : ✓')
print('✅ Step 3 import check passed.')


Created symlink alias: /content/cascade_guard -> /content/cascade_gaurd_openEnv
cascade_guard package path             : /content/cascade_guard/__init__.py
Import strategy used                  : symlink-alias
from cascade_guard.models             : ✓
from cascade_guard.tasks              : ✓
from cascade_guard.server.cascade_env : ✓
✅ Step 3 import check passed.


## 🗺️ Step 4 — Pre-warm OSM data (download once, cached forever)

First call downloads the city graph; subsequent calls use the local cache.

In [5]:
import os, time
from pathlib import Path

OSM_CACHE = '/content/osmnx_cache'
os.makedirs(OSM_CACHE, exist_ok=True)

try:
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.cache_folder = OSM_CACHE
    print(f"osmnx {ox.__version__} — cache: {OSM_CACHE}")
except ImportError:
    print("osmnx not found — environment will use its own cache.")

try:
    from cascade_guard.models import CascadeAction
    from cascade_guard.server.cascade_environment import CascadeEnvironment
    from cascade_guard.tasks import TASK_CONFIGS, TASK_SEED_SPLITS

    TASKS = list(TASK_SEED_SPLITS.keys())
    print(f"\nTasks found: {TASKS}")
    print("Pre-warming CascadeEnvironment ...")
    t0 = time.perf_counter()

    for task_id in TASKS[:3]:
        env = CascadeEnvironment()
        seed = TASK_SEED_SPLITS[task_id]['train'][0]
        obs = env.reset(task_id=task_id, seed=seed,
                        reward_mode='grpo', training_mode=True)
        env.step(CascadeAction(action_type='wait', target_node_id=None))
        print(f"  {task_id:30s} nodes={len(obs.nodes)}  ✓")
        del env

    print(f"\n✅ OSM pre-warm complete in {time.perf_counter()-t0:.1f}s")
except Exception as e:
    print(f"⚠️  Pre-warm raised: {e}")
    print("Training will still proceed; first steps may be slower.")

osmnx 2.1.0 — cache: /content/osmnx_cache

Tasks found: ['task_easy', 'task_medium', 'task_hard', 'task_gen_blackout', 'task_cyberattack', 'task_surge_demand', 'task_real_city', 'task_osm_london', 'task_osm_mumbai', 'task_osm_bangalore', 'task_osm_delhi', 'task_osm_nyc', 'task_osm_tokyo']
Pre-warming CascadeEnvironment ...
  task_easy                      nodes=6  ✓
  task_medium                    nodes=12  ✓
  task_hard                      nodes=15  ✓

✅ OSM pre-warm complete in 0.0s


## ⚙️ Step 5 — Training configuration

Tuned for **T4 (15 GB VRAM)** with 4-bit BitsAndBytes quantization + LoRA.  
Defaults are set for a quick proof-of-learning run (~15–25 min on T4).

In [ ]:
import os, torch

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'   # ~1 GB download; fast on T4 with 4-bit

# ── SFT dataset path ───────────────────────────────────────────────────────────
# Upload cascadeguard_sft_dataset.csv to Colab or place it in the repo dir.
SFT_DATASET_PATH = '/content/cascadeguard_sft_dataset.csv'
# Fallback: also check repo dir
import os
if not os.path.exists(SFT_DATASET_PATH):
    _alt = '/content/cascade_gaurd_openEnv/cascadeguard_sft_dataset.csv'
    if os.path.exists(_alt):
        SFT_DATASET_PATH = _alt
        print(f"SFT dataset found at: {SFT_DATASET_PATH}")

# ── Training ────────────────────────────────────────────────────────────────────
SFT_STEPS        = 200    # SFT warm-start steps (increase for better coverage)
GRPO_STEPS       = 150    # GRPO steps (increase for more RL signal)
STATES           = 64     # Training states for GRPO (increase for quality)
GENERATIONS      = 4      # GRPO group size — minimum 4
LORA_R           = 8      # LoRA rank (8 fits comfortably on T4 with 4-bit)
LR_SFT           = 2e-5   # SFT learning rate
LR_GRPO          = 8e-6   # GRPO learning rate
MAX_SEQ_LEN      = 768    # Reduced to fit T4 VRAM with 4-bit
MAX_COMP_LEN     = 200    # Max tokens for GRPO generation

# ── Paths ────────────────────────────────────────────────────────────────────
REPO_DIR      = '/content/cascade_gaurd_openEnv'
OUTPUT_DIR    = f'{REPO_DIR}/training_outputs'
CURVES_DIR    = f'{REPO_DIR}/training_curves'
RESULTS_CSV   = f'{REPO_DIR}/cascadeguard_eval_results.csv'
CACHE_DIR     = '/root/.cache/cascadeguard_hf'

# ── Mode flags ─────────────────────────────────────────────────────────────────
SMOKE_TEST = False   # True → 5 SFT steps, 10 GRPO steps (~2 min sanity check)
RUN_EVAL   = True

for d in [OUTPUT_DIR, CURVES_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── T4 environment flags ───────────────────────────────────────────────────────
os.environ['TORCHINDUCTOR_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
os.environ['HF_HOME']               = CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE_DIR
os.environ['TRANSFORMERS_CACHE']    = CACHE_DIR
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.verbose         = False

if SMOKE_TEST:
    SFT_STEPS, GRPO_STEPS, STATES = 5, 10, 16
    print("⚠️  SMOKE TEST — minimal steps only")

print(f"Model            : {MODEL}")
print(f"SFT steps        : {SFT_STEPS}  |  GRPO steps : {GRPO_STEPS}")
print(f"GRPO states      : {STATES}     |  Generations: {GENERATIONS}")
print(f"LoRA r           : {LORA_R}     |  Max seq len: {MAX_SEQ_LEN}")
print(f"SFT dataset      : {SFT_DATASET_PATH}")
print(f"Output dir       : {OUTPUT_DIR}")

Model            : Qwen/Qwen2.5-0.5B-Instruct
SFT steps        : 200  |  GRPO steps : 150
GRPO states      : 64     |  Generations: 4
LoRA r           : 8     |  Max seq len: 768
SFT dataset      : /content/cascadeguard_sft_dataset.csv
Output dir       : /content/cascade_gaurd_openEnv/training_outputs


## 📊 Step 6 — Load SFT dataset from CSV

Formats the pre-built `cascadeguard_sft_dataset.csv` into chat-template format
using the model's own tokenizer. The `completion` column already contains
`<think>...</think>\nACTION TARGET` formatted responses.

In [9]:
import pandas as pd
from datasets import Dataset

# ── Load CSV ───────────────────────────────────────────────────────────────────
sft_raw = pd.read_csv(SFT_DATASET_PATH)
print(f"SFT dataset loaded: {len(sft_raw)} rows")
print(f"Columns: {sft_raw.columns.tolist()}")
print(f"\nTask distribution:")
print(sft_raw['task'].value_counts().to_string())
print(f"\nPhase distribution:")
print(sft_raw['phase'].value_counts().to_string())

# Drop rows with null completions
sft_raw = sft_raw.dropna(subset=['system', 'prompt', 'completion'])
print(f"\nAfter dropping nulls: {len(sft_raw)} rows")

# ── Preview ────────────────────────────────────────────────────────────────────
print("\nSample completion (first 300 chars):")
print(repr(sft_raw['completion'].iloc[0][:300]))

SFT dataset loaded: 155 rows
Columns: ['id', 'task', 'seed', 'step', 'phase', 'system', 'prompt', 'thinking', 'completion', 'action', 'target', 'episode_score', 'reward_step', 'sector_health_power', 'sector_health_water', 'sector_health_hospital', 'sector_health_telecom', 'cascade_depth', 'budget_remaining', 'steps_taken', 'max_steps']

Task distribution:
task
task_hard           50
task_medium         40
task_easy           25
task_blackout       25
task_cyberattack    15

Phase distribution:
phase
mid      63
late     52
early    40

After dropping nulls: 155 rows

Sample completion (first 300 chars):
'<think>Current situation: cascade_depth=1, budget=$7,520, steps_remaining=4. Critical nodes: BATT_BANK. These need immediate attention. Hospital sector average health: 100%. Hospital survival is highest priority. Power sector average: 85%. Power failures cascade to hospital and water. Analyzing targ'


## 🧠 Step 7 — Load model (4-bit BitsAndBytes + LoRA via PEFT)

No Unsloth. Uses standard HuggingFace `transformers` + `bitsandbytes` + `peft`.

In [10]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print(f"Loading tokenizer for {MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)
# Ensure pad token is set (required for batched training)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'   # SFT needs right-padding
print("Tokenizer loaded ✓")

# ── 4-bit quantization config ──────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL} in 4-bit ...")
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16,
)
print(f"Base model loaded in {time.perf_counter()-t0:.0f}s ✓")

# ── Prepare for k-bit training (required for 4-bit + gradient checkpointing) ──
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# ── Apply LoRA ─────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

# ── T4 mixed-precision safety pass ─────────────────────────────────────────────
# On T4 we train with fp16=True. PEFT/Qwen sometimes leaves LoRA adapter weights
# in bf16 (the model's native dtype), which makes torch's fp16 GradScaler crash:
#   NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'
# Force every trainable param to fp32 (standard practice for 4-bit + LoRA + fp16),
# and force non-trainable, non-quantized params (embed / layernorm / lm_head) to
# fp16 so autocast stays consistent. Quantized 4-bit params are uint8 storage
# under the hood and must NOT be touched.
import bitsandbytes as bnb  # noqa: F401  (already installed for 4-bit quant)
from bitsandbytes.nn import Params4bit, Int8Params

bf16_trainable = 0
fp32_trainable = 0
for name, param in model.named_parameters():
    if isinstance(param, (Params4bit, Int8Params)):
        continue  # leave quantized weights alone
    if param.requires_grad:
        if param.dtype != torch.float32:
            param.data = param.data.to(torch.float32)
            bf16_trainable += 1
        else:
            fp32_trainable += 1
    else:
        # Keep non-trainable fp tensors in fp16 to match AMP autocast on T4
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)

print(f"Trainable params recast to fp32: {bf16_trainable} (already fp32: {fp32_trainable})")
model.print_trainable_parameters()

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM used after model load: {used:.1f} / {total:.1f} GB")

print("\n✅ Model ready for training (fp32 LoRA adapters, fp16 base for T4).")

Loading tokenizer for Qwen/Qwen2.5-0.5B-Instruct ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Tokenizer loaded ✓
Loading Qwen/Qwen2.5-0.5B-Instruct in 4-bit ...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded in 13s ✓
Trainable params recast to fp32: 0 (already fp32: 192)
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
VRAM used after model load: 0.7 / 15.6 GB

✅ Model ready for training (fp32 LoRA adapters, fp16 base for T4).


## 🔥 Step 8 — SFT Training (TRL SFTTrainer)

Trains on `cascadeguard_sft_dataset.csv` using supervised next-token prediction.
This teaches the model all 16 action formats and `<think>...</think>` reasoning
before GRPO refines the policy with reward signal.

In [11]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# ── Format rows into chat template ─────────────────────────────────────────────
def format_sft_row(row):
    """Convert a CSV row into a single formatted string using the model's chat template."""
    messages = [
        {'role': 'system',    'content': str(row['system']).strip()},
        {'role': 'user',      'content': str(row['prompt']).strip()},
        {'role': 'assistant', 'content': str(row['completion']).strip()},
    ]
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        # Fallback: manual format if apply_chat_template fails
        text = (
            f"<|system|>\n{messages[0]['content']}\n"
            f"<|user|>\n{messages[1]['content']}\n"
            f"<|assistant|>\n{messages[2]['content']}"
        )
    return {'text': text}

print("Formatting SFT dataset ...")
sft_records = [format_sft_row(row) for _, row in sft_raw.iterrows()]
sft_dataset = Dataset.from_list(sft_records)
print(f"SFT dataset ready: {len(sft_dataset)} examples")
print(f"Sample (first 300 chars):")
print(sft_dataset[0]['text'][:300])

# ── SFT training config ────────────────────────────────────────────────────────
sft_args = SFTConfig(
    output_dir                  = f'{OUTPUT_DIR}/sft',
    max_steps                   = SFT_STEPS,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = LR_SFT,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    max_grad_norm               = 1.0,
    logging_steps               = 1,
    save_steps                  = max(SFT_STEPS, 1),
    report_to                   = 'none',
    bf16                        = False,   # T4 does NOT support bf16
    fp16                        = True,    # T4 → fp16
    dataset_text_field          = 'text',
    packing                     = False,
    gradient_checkpointing_kwargs={"use_reentrant": False},# simpler; set True for speed if no issues
)

# ── Metric logger for SFT ──────────────────────────────────────────────────────
from transformers import TrainerCallback

class SFTMetricLogger(TrainerCallback):
    def __init__(self):
        self.records = []
    def on_log(self, _args, state, _control, logs=None, **kwargs):
        if logs:
            step = state.global_step
            rec = {'step': step}
            for k, out in [('loss','loss'), ('train/loss','loss'), ('learning_rate','lr')]:
                if k in logs and out not in rec:
                    rec[out] = float(logs[k]) if logs[k] is not None else float('nan')
            if len(rec) > 1:
                self.records.append(rec)

sft_logger = SFTMetricLogger()

# ── Last-mile dtype safety net for T4 + fp16 ──────────────────────────────────
# Some PEFT / TRL versions silently re-cast LoRA adapters back to bf16 between
# get_peft_model() and trainer.train(). Re-assert fp32 on trainable params here
# so the fp16 GradScaler never sees a bf16 gradient.
_recast = 0
for _n, _p in model.named_parameters():
    if _p.requires_grad and _p.dtype != torch.float32:
        _p.data = _p.data.to(torch.float32)
        _recast += 1
if _recast:
    print(f"[T4 safety] re-cast {_recast} trainable params to fp32 before SFT.")

sft_trainer = SFTTrainer(
    model            = model,
    args             = sft_args,
    train_dataset    = sft_dataset,
    processing_class = tokenizer,
    callbacks        = [sft_logger],
)
# ── T4 Last-Mile BF16→FP32 cast ──────────────────────────────────────────
# SFTTrainer.__init__ may silently re-cast LoRA adapters back to BF16.
# Re-assert fp32 on ALL trainable params immediately before .train().
from bitsandbytes.nn import Params4bit, Int8Params

_recast = 0
for _n, _p in model.named_parameters():
    if isinstance(_p, (Params4bit, Int8Params)):
        continue                        # never touch quantized storage
    if _p.requires_grad and _p.dtype != torch.float32:
        _p.data = _p.data.to(torch.float32)
        _recast += 1
    elif not _p.requires_grad and _p.dtype == torch.bfloat16:
        _p.data = _p.data.to(torch.float16)  # non-trainable: match AMP autocast
if _recast:
    print(f"[T4 cast] {_recast} trainable params → fp32 just before sft_trainer.train()")

print(f"\nStarting SFT training: {SFT_STEPS} steps ...")
sft_trainer.train()


sft_save = f'{OUTPUT_DIR}/sft_final'
sft_trainer.save_model(sft_save)
tokenizer.save_pretrained(sft_save)
print(f"\n✅ SFT checkpoint saved → {sft_save}")

if torch.cuda.is_available():
    print(f"   Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Formatting SFT dataset ...
SFT dataset ready: 155 examples
Sample (first 300 chars):
<|im_start|>system
You are CascadeGuard, an AI agent managing critical infrastructure resilience. Your job is to detect and contain cascade failures across power, water, hospital, and telecom systems. Think step by step before acting. Consider cascade depth, node centrality, sector dependencies, ava


Adding EOS to train dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[T4 cast] 192 trainable params → fp32 just before sft_trainer.train()

Starting SFT training: 200 steps ...


Step,Training Loss
1,2.229005
2,2.199629
3,2.217932
4,2.242988
5,2.243701
6,2.217518
7,2.224084
8,2.175347
9,2.223497
10,2.162116


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]


✅ SFT checkpoint saved → /content/cascade_gaurd_openEnv/training_outputs/sft_final
   Peak VRAM: 3.99 GB


## 📊 Step 9 — Load `train_grpo` helpers & collect GRPO training states

States are gathered by rolling out teacher policies against the **live** `CascadeEnvironment`.

In [ ]:
import importlib.util, sys, os, argparse, time
from pathlib import Path
from datasets import Dataset

TRAIN_GRPO_PATH = '/content/cascade_gaurd_openEnv/training/train_grpo.py'

# ── Load train_grpo module (patch parse_args so it ignores Colab argv) ─────────
_orig_parse = argparse.ArgumentParser.parse_args
argparse.ArgumentParser.parse_args = lambda self, args=None, ns=None: _orig_parse(self, [], ns)

spec = importlib.util.spec_from_file_location('train_grpo', TRAIN_GRPO_PATH)
train_grpo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_grpo)

argparse.ArgumentParser.parse_args = _orig_parse   # restore
print("✅ train_grpo helpers loaded.")

# ── Build args namespace ────────────────────────────────────────────────────────
TASKS = [
    'task_easy', 'task_medium', 'task_surge_demand',
    'task_gen_blackout', 'task_hard', 'task_cyberattack',
]

args = argparse.Namespace(
    model                      = MODEL,
    no_unsloth                 = True,       # always True — no Unsloth in this notebook
    hf_token                   = os.environ.get('HF_TOKEN', ''),
    cache_dir                  = CACHE_DIR,
    lora_r                     = LORA_R,
    lora_alpha                 = LORA_R * 2,
    steps                      = GRPO_STEPS,
    states                     = STATES,
    generations                = GENERATIONS,
    lr                         = LR_GRPO,
    max_seq_len                = MAX_SEQ_LEN,
    max_completion_len         = MAX_COMP_LEN,
    sft_warmstart_steps        = 0,          # SFT already done in Step 8
    sft_warmstart_lr           = 0.0,
    action_only_prompts        = True,
    debug_parseability         = False,
    debug_parseability_samples = 8,
    debug_hard_rollouts        = False,
    debug_hard_rollout_seeds   = 1,
    reward_stats_every         = 10,
    tasks                      = TASKS,
    eval_tasks                 = TASKS,
    eval_seeds                 = 2,
    output_dir                 = OUTPUT_DIR,
    results_csv                = RESULTS_CSV,
    resume_from                = None,
    smoke_test                 = SMOKE_TEST,
    no_eval                    = not RUN_EVAL,
    repo_root                  = '/content/cascade_gaurd_openEnv',
)

# ── Import cascade symbols via train_grpo ──────────────────────────────────────
project_root = Path('/content/cascade_gaurd_openEnv')
os.chdir(project_root)

(
    CascadeAction, CascadeEnvironment, TASK_CONFIGS,
    TASK_SEED_SPLITS, build_system_prompt, build_user_prompt,
    make_training_prompt_fn, parse_action_from_response,
) = train_grpo.import_cascade(project_root)

print("Cascade symbols imported ✓")

# ── Sanity check ───────────────────────────────────────────────────────────────
train_grpo.run_env_sanity_check(
    CascadeEnvironment, CascadeAction, TASK_SEED_SPLITS, args.tasks
)
print("\n✅ All tasks passed environment sanity check.")

# ── Collect training states — BALANCED per-task ────────────────────────────────
# collect_training_states() fills its quota sequentially: task_easy finishes
# first (5 seeds × 10 steps ≈ 50 rows) and exhausts STATES=64 before the
# harder tasks ever get sampled.  Fix: call once per task with a per-task cap,
# then shuffle so GRPO never sees streaks of one task type.
import random as _sampling_rng

heuristic_policy = train_grpo.make_heuristic_policy(CascadeAction)

states_per_task = max(STATES // len(TASKS), 8)   # e.g. 120 // 6 = 20 each
remainder       = STATES - states_per_task * len(TASKS)  # top-up after main loop

print(f"\nCollecting {STATES} states — {states_per_task}/task × {len(TASKS)} tasks (+ {remainder} extra) ...")
t0 = time.perf_counter()

rows = []
for _task in TASKS:
    _task_rows = train_grpo.collect_training_states(
        num_states              = states_per_task,
        train_tasks             = [_task],
        TASK_SEED_SPLITS        = TASK_SEED_SPLITS,
        CascadeAction           = CascadeAction,
        CascadeEnvironment      = CascadeEnvironment,
        heuristic_policy        = heuristic_policy,
        make_training_prompt_fn = make_training_prompt_fn,
        action_only_prompts     = args.action_only_prompts,
    )
    rows.extend(_task_rows)
    print(f"  {_task:30s}  {len(_task_rows)} states collected")

# Top-up remainder from hardest tasks (ensures total == STATES exactly)
if remainder > 0:
    hard_tasks = ['task_hard', 'task_cyberattack', 'task_gen_blackout']
    _extra = train_grpo.collect_training_states(
        num_states              = remainder,
        train_tasks             = hard_tasks,
        TASK_SEED_SPLITS        = TASK_SEED_SPLITS,
        CascadeAction           = CascadeAction,
        CascadeEnvironment      = CascadeEnvironment,
        heuristic_policy        = heuristic_policy,
        make_training_prompt_fn = make_training_prompt_fn,
        action_only_prompts     = args.action_only_prompts,
    )
    rows.extend(_extra)
    print(f"  [top-up hard tasks]           {len(_extra)} states collected")

# Shuffle so GRPO batches mix tasks — prevents single-task reward-collapse
_sampling_rng.Random(42).shuffle(rows)

elapsed = time.perf_counter() - t0

grpo_train_ds = Dataset.from_list(rows)
print(f"\nDataset  : {grpo_train_ds}")
print(f"Collected: {len(rows)} states in {elapsed:.1f}s")

from collections import Counter
dist = Counter(r['task_id'] for r in rows)
print("\nTask distribution:")
for t, n in sorted(dist.items()):
    bar = '█' * (n // 2)
    print(f"  {t:30s}  {n:3d} states  {bar}")

expected_per_task = states_per_task
coverage = sum(1 for t in TASKS if dist.get(t, 0) >= expected_per_task * 0.8)
print(f"\nTask coverage: {coverage}/{len(TASKS)} tasks have ≥{int(expected_per_task*0.8)} states")
if coverage < len(TASKS):
    print("  ⚠️  Some tasks under-sampled — consider increasing STATES")

# ── Build reward function ──────────────────────────────────────────────────────
reward_fn = train_grpo.make_grpo_reward_fn(
    CascadeEnvironment, CascadeAction, parse_action_from_response, args=args,
)

# ── Reward smoke test ──────────────────────────────────────────────────────────
test_r = reward_fn(
    prompts             = [rows[0]['prompt']],
    completions         = ['<action>harden(POWER_GEN_1)</action>'],
    task_id             = [rows[0]['task_id']],
    seed                = [rows[0]['seed']],
    history_json        = [rows[0]['history_json']],
    teacher_action_json = [rows[0].get('teacher_action_json', '{}')],
    teacher_policy      = [rows[0].get('teacher_policy', 'advanced_crisis_manager')],
)
print(f"\nReward smoke test : {test_r}  ✓")

✅ train_grpo helpers loaded.
Cascade symbols imported ✓

✅ All tasks passed environment sanity check.

Dataset  : Dataset({
    features: ['prompt', 'task_id', 'seed', 'teacher_policy', 'history_json', 'teacher_action_json', 'teacher_action_text'],
    num_rows: 64
})
Collected: 64 states in 0.1s

Task distribution:
  task_easy                       50 states
  task_medium                     14 states

Reward smoke test : [0.7502753623188403]  ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 🧪 Cell A — Pre-training signal sanity (does the env / grader / teacher
# actually contain learnable signal?)
# ══════════════════════════════════════════════════════════════════════════════
# Two checks, both run BEFORE GRPO so we never burn GPU on a dead pipeline:
#
#   (1) Grader sensitivity:
#       For 8 sampled states, enumerate all legal actions and step a fresh
#       env clone with each. Compute state.score after each step. If the
#       max-min spread is > 0.01 on at least 50% of states, the grader can
#       distinguish actions and score_delta is a usable (small) signal.
#
#   (2) Reward separation (heuristic vs random_legal):
#       Roll each policy on 10 episodes per task. If the heuristic's mean
#       episode reward is >= 1.5x the random policy's mean reward, the env
#       reward itself is well-shaped and there IS something to optimize.
#
# Failures here are upstream of training and cannot be fixed by tuning GRPO.
# ══════════════════════════════════════════════════════════════════════════════
import time as _time_a, statistics as _stats_a, json as _json_a, random as _rng_a

print("\n" + "═" * 72)
print(" Cell A  ·  Pre-training signal-sanity diagnostics")
print("═" * 72)

# ── (1) Grader sensitivity ───────────────────────────────────────────────────
print("\n[A.1] Grader sensitivity: stepping each legal action from 8 sampled states ...")

def _replay_history(_env, _hist):
    if isinstance(_hist, str):
        try:
            _hist = _json_a.loads(_hist or '[]')
        except Exception:
            _hist = []
    if not isinstance(_hist, list):
        return
    for _item in _hist:
        if not isinstance(_item, dict):
            continue
        try:
            _env.step(CascadeAction(**_item))
        except Exception:
            return

_NUM_PROBE = min(8, len(rows))
_probe_idxs = _rng_a.Random(99).sample(range(len(rows)), _NUM_PROBE)
_score_spreads = []
_pass_count = 0

for _pi, _row_idx in enumerate(_probe_idxs):
    _row = rows[_row_idx]
    _env_probe = CascadeEnvironment()
    _obs_probe = _env_probe.reset(
        task_id=str(_row['task_id']), seed=int(_row['seed']),
        scenario_split='train', reward_mode='grpo', training_mode=True,
    )
    _replay_history(_env_probe, _row.get('history_json', '[]'))
    try:
        _legal = _env_probe.get_legal_actions() or []
    except Exception:
        _legal = []
    if not _legal:
        print(f"  probe[{_pi}] task={_row['task_id']:20s}  no legal actions, skipping.")
        continue

    _scores = []
    for _la in _legal[:8]:                      # cap to 8 actions per state for speed
        _env_clone = CascadeEnvironment()
        _env_clone.reset(
            task_id=str(_row['task_id']), seed=int(_row['seed']),
            scenario_split='train', reward_mode='grpo', training_mode=True,
        )
        _replay_history(_env_clone, _row.get('history_json', '[]'))
        try:
            _act_obj = CascadeAction(
                action_type    = _la.get('action_type', 'wait'),
                target_node_id = _la.get('target_node_id'),
                parameters     = _la.get('parameters') or {},
            )
            _env_clone.step(_act_obj)
            _scores.append(float(_env_clone.state.score))
        except Exception:
            continue

    if len(_scores) >= 2:
        _spread = max(_scores) - min(_scores)
        _score_spreads.append(_spread)
        _pass = _spread > 0.01
        _pass_count += int(_pass)
        print(f"  probe[{_pi}] task={_row['task_id']:20s}  "
              f"actions={len(_scores):2d}  "
              f"score range=[{min(_scores):.4f}, {max(_scores):.4f}]  "
              f"spread={_spread:.4f}  {'PASS' if _pass else 'FAIL'}")
    else:
        print(f"  probe[{_pi}] task={_row['task_id']:20s}  insufficient legal actions ({len(_scores)})")

if _score_spreads:
    _mean_spread = _stats_a.mean(_score_spreads)
    _max_spread  = max(_score_spreads)
    print(f"\n  mean spread: {_mean_spread:.4f}   max spread: {_max_spread:.4f}")
    print(f"  passing probes (spread > 0.01): {_pass_count}/{len(_score_spreads)}")
    if _pass_count >= max(len(_score_spreads) * 0.5, 1):
        print("  ✅ A.1 PASS — grader can distinguish actions; score_delta is usable.")
    else:
        print("  ⚠️  A.1 WARN — grader is INSENSITIVE; score_delta cannot drive learning.")
        print("      The redesigned reward (step-reward dominant) compensates,")
        print("      but consider revisiting graders.py weights / time-aggregates.")
else:
    print("  ⚠️  A.1 SKIPPED — no probes had >= 2 legal actions.")

# ── (2) Reward separation (heuristic vs random) ──────────────────────────────
print("\n[A.2] Reward separation: heuristic vs random_legal on 10 episodes per task ...")

def _roll_episode(_policy_fn, _task, _seed):
    _env = CascadeEnvironment()
    _obs = _env.reset(task_id=_task, seed=int(_seed),
                      scenario_split='train', reward_mode='grpo', training_mode=False)
    _total = 0.0
    for _step in range(int(TASK_CONFIGS[_task]['max_steps'])):
        try:
            _legal = _env.get_legal_actions() or []
        except Exception:
            _legal = []
        try:
            _act = _policy_fn(_obs, _legal)
        except TypeError:
            _act = _policy_fn(_obs)
        except Exception:
            _act = CascadeAction(action_type='wait', target_node_id=None)
        _obs = _env.step(_act)
        _total += float(getattr(_obs, 'reward', 0.0) or 0.0)
        if getattr(_obs, 'done', False):
            break
    return _total

def _random_policy(_obs, _legal):
    if not _legal:
        return CascadeAction(action_type='wait', target_node_id=None)
    _l = _rng_a.Random().choice(_legal)
    return CascadeAction(
        action_type    = _l.get('action_type', 'wait'),
        target_node_id = _l.get('target_node_id'),
        parameters     = _l.get('parameters') or {},
    )

_heur = train_grpo.make_heuristic_policy(CascadeAction)
_NUM_EPS = 10 if not SMOKE_TEST else 4
_seeds_a = list(range(701, 701 + _NUM_EPS))

_a2_summary = []
_a2_pass = 0
for _task in TASKS:
    _t0 = _time_a.perf_counter()
    _h_eps = [_roll_episode(_heur, _task, _s) for _s in _seeds_a]
    _r_eps = [_roll_episode(_random_policy, _task, _s) for _s in _seeds_a]
    _h_mu = _stats_a.mean(_h_eps)
    _r_mu = _stats_a.mean(_r_eps)
    _ratio = _h_mu / _r_mu if abs(_r_mu) > 1e-3 else float('inf') if _h_mu > 0 else 0.0
    _ok = (_h_mu - _r_mu) > 0.5 or _ratio >= 1.5
    _a2_pass += int(_ok)
    _a2_summary.append((_task, _h_mu, _r_mu, _ratio, _ok))
    print(f"  {_task:25s}  heur μ={_h_mu:+7.3f}   random μ={_r_mu:+7.3f}   "
          f"diff={_h_mu - _r_mu:+6.3f}   {'PASS' if _ok else 'WARN'}   "
          f"({_time_a.perf_counter()-_t0:5.1f}s)")

print(f"\n  tasks where heuristic clearly beats random: {_a2_pass}/{len(TASKS)}")
if _a2_pass >= max(len(TASKS) * 0.5, 1):
    print("  ✅ A.2 PASS — env reward is well-shaped; there IS something to optimize.")
else:
    print("  ❌ A.2 FAIL — heuristic does not beat random; the env reward is")
    print("      either too noisy or too sparse for any RL algorithm to learn.")
    print("      STOP and revisit cascade_environment.py::_compute_reward")
    print("      before running GRPO.")

print("\n" + "─" * 72)
print(" Cell A complete. Continuing to GRPO training setup ↓")
print("─" * 72)


## 🚂 Step 10 — GRPO Training (TRL GRPOTrainer)

Pure TRL GRPO — no Unsloth, no MergeKit.  
`fp16=True, bf16=False` — T4 does **not** support bf16.

In [15]:
!pip install weave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 91.4 MB/s eta 0:00:00


In [16]:
import sys, importlib

# 1. Purge the stale weave and trl trainer modules from the module cache
to_purge = [k for k in sys.modules if k == "weave" or k.startswith("weave.")
                                    or k.startswith("trl.trainer")]
for k in to_purge:
    del sys.modules[k]
print(f"Purged {len(to_purge)} stale modules: {to_purge[:6]} ...")

# 2. Verify the new weave has EvaluationLogger
import weave
print(f"weave version : {weave.__version__}")
print(f"EvaluationLogger present: {hasattr(weave, 'EvaluationLogger')}")

# 3. Now test that trl imports cleanly
from trl import GRPOTrainer, GRPOConfig
print("✅ TRL imported cleanly — ready for GRPO cell")

Purged 0 stale modules: [] ...
weave version : 0.52.37
EvaluationLogger present: True
✅ TRL imported cleanly — ready for GRPO cell


In [ ]:
import os, torch
from trl import GRPOTrainer, GRPOConfig
from pathlib import Path
from transformers import TrainerCallback


# ── Ensure model is in train mode after SFT ────────────────────────────────────
model.train()

# Verify no frozen param issues after SFT
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# ── Metric logger ──────────────────────────────────────────────────────────────
class GRPOMetricLogger(TrainerCallback):
    def __init__(self):
        self.records = []
    def on_log(self, _args, state, _control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        rec = {'step': step}
        for key, out_key in [
            ('loss',          'loss'),
            ('train/loss',    'loss'),
            ('reward',        'reward'),
            ('train/reward',  'reward'),
            ('rewards',       'reward'),
            ('mean_reward',   'reward'),
            ('grad_norm',     'grad_norm'),
            ('learning_rate', 'lr'),
        ]:
            if key in logs and out_key not in rec:
                v = logs[key]
                rec[out_key] = float(v) if v is not None else float('nan')
        if len(rec) > 1:
            self.records.append(rec)

grpo_logger = GRPOMetricLogger()

# ── GRPOConfig ─────────────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Older TRL versions don't accept top_p / top_k / scale_rewards / beta /
# epsilon as GRPOConfig kwargs. We build a kwargs dict, then drop any keys
# the installed TRL version doesn't know about so the cell stays portable.
import inspect as _gc_inspect
_gc_sig = _gc_inspect.signature(GRPOConfig.__init__).parameters
_gc_kwargs = dict(
    output_dir                  = OUTPUT_DIR,
    max_steps                   = GRPO_STEPS,
    learning_rate               = LR_GRPO,
    per_device_train_batch_size = GENERATIONS,
    gradient_accumulation_steps = 2,
    num_generations             = GENERATIONS,
    max_completion_length       = MAX_COMP_LEN,
    # Sampling: higher temp + nucleus sampling so the 0.5B SFT model
    # produces diverse completions in each GRPO group. Without diversity
    # all rewards in a group are equal -> advantage = 0 -> zero gradient.
    temperature                 = 1.0,
    top_p                       = 0.95,
    top_k                       = 50,
    # CRITICAL: TRL's default scale_rewards="group" divides advantages by
    # within-group reward std. When rewards are nearly equal (our failure
    # mode), std -> 0 and advantages collapse to 0/NaN -> masked gradient
    # -> flat loss. Disabling lets small absolute differences flow through.
    scale_rewards               = False,
    # Small KL anchor against the reference model -- prevents catastrophic
    # policy collapse and keeps exploration grounded.
    beta                        = 0.04,
    epsilon                     = 0.2,
    warmup_ratio                = 0.08,
    lr_scheduler_type           = 'cosine',
    max_grad_norm               = 0.30,
    logging_steps               = 1,
    save_steps                  = max(GRPO_STEPS, 1),
    report_to                   = 'none',
    remove_unused_columns       = False,
    bf16                        = False,   # T4 does NOT support bf16
    fp16                        = True,    # T4 -> fp16
)
_filtered_kwargs = {k: v for k, v in _gc_kwargs.items() if k in _gc_sig}
_dropped = sorted(set(_gc_kwargs) - set(_filtered_kwargs))
if _dropped:
    print(f"[GRPOConfig] dropped kwargs unsupported by this TRL version: {_dropped}")
    print(f"             you may want to upgrade TRL: pip install -U 'trl>=0.24.0'")

training_args = GRPOConfig(**_filtered_kwargs)

# ── Compatibility patch for older TRL versions ─────────────────────────────────
if hasattr(train_grpo, '_ensure_trainer_compatibility'):
    train_grpo._ensure_trainer_compatibility(model)

trainer = GRPOTrainer(
    model            = model,
    args             = training_args,
    reward_funcs     = reward_fn,
    train_dataset    = grpo_train_ds,
    processing_class = tokenizer,
    peft_config      = None,   # LoRA already applied in Step 7
    callbacks        = [grpo_logger],
)

print(f"Starting GRPO training: {GRPO_STEPS} steps × {GENERATIONS} generations ...")
# ── T4 Last-Mile BF16→FP32 cast ───────────────────────────────────────────────
# GRPOTrainer.__init__ (above) may silently re-cast LoRA adapters to BF16.
# This MUST be the last operation before trainer.train().
_recast = 0
for _n, _p in model.named_parameters():
    if isinstance(_p, (Params4bit, Int8Params)):
        continue                              # never touch quantized storage
    if _p.requires_grad and _p.dtype != torch.float32:
        _p.data = _p.data.to(torch.float32)
        _recast += 1
    elif not _p.requires_grad and _p.dtype == torch.bfloat16:
        _p.data = _p.data.to(torch.float16)  # non-trainable: align with fp16 autocast
if _recast:
    print(f"[T4 cast] {_recast} trainable params re-cast → fp32 after GRPOTrainer init")
else:
    print("[T4 cast] All trainable params already fp32 ✓")

# ══════════════════════════════════════════════════════════════════════════════
# 🧪 Cell B — Pre-training diversity check (model + dataset interaction)
# ══════════════════════════════════════════════════════════════════════════════
# Before burning 30+ minutes of GPU on GRPO, verify two pipeline preconditions:
#   1. The model produces DIVERSE completions at the chosen temperature
#      (otherwise within-group reward σ = 0 → no learning).
#   2. The model does NOT just copy the recorded teacher action
#      (otherwise score_delta = 0 by construction → no learning).
#
# Pass thresholds:
#   - mean_unique_actions / num_generations  >= 0.40
#   - agent==teacher rate                     <= 0.70
#
# If either fails → raise temperature, shorten SFT, or rely more on the
# in-reward exploration bonus (cascade_grpo_reward in train_grpo.py).
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 72)
print(" Cell B  ·  Pre-training diversity & teacher-imitation check")
print("═" * 72)

import json as _json_b
_GEN_N = max(int(GENERATIONS), 4)
_NUM_PROBE_ROWS = min(8, len(grpo_train_ds))
_probe_rows = grpo_train_ds.shuffle(seed=1234).select(range(_NUM_PROBE_ROWS))

_unique_ratios = []
_teacher_match_count = 0
_total_completions = 0
_parseable_count = 0

for _i, _row in enumerate(_probe_rows):
    _enc = tokenizer(
        _row['prompt'], return_tensors='pt',
        truncation=True, max_length=MAX_SEQ_LEN,
    ).to(model.device)
    with torch.no_grad():
        _out = model.generate(
            **_enc,
            max_new_tokens       = MAX_COMP_LEN,
            do_sample            = True,
            temperature          = 1.0,
            top_p                = 0.95,
            top_k                = 50,
            num_return_sequences = _GEN_N,
            pad_token_id         = tokenizer.eos_token_id,
        )
    _texts = [
        tokenizer.decode(
            _out[k][_enc['input_ids'].shape[1]:], skip_special_tokens=True,
        ) for k in range(_out.shape[0])
    ]
    # Parse each completion → CascadeAction → key string
    _action_keys = []
    _parseable_n = 0
    for _t in _texts:
        try:
            _act = parse_action_from_response(_t, None)
            _action_keys.append(f"{_act.action_type}({_act.target_node_id})")
            _parseable_n += 1
        except Exception:
            _action_keys.append("__UNPARSEABLE__")
    # Teacher action key
    try:
        _t_payload = _row.get('teacher_action_json', '{}')
        _t_obj = _json_b.loads(_t_payload) if isinstance(_t_payload, str) else _t_payload
        _t_key = f"{_t_obj.get('action_type')}({_t_obj.get('target_node_id')})"
    except Exception:
        _t_key = '?'

    _unique = len(set(_action_keys))
    _ratio = _unique / max(len(_action_keys), 1)
    _matches = sum(1 for k in _action_keys if k == _t_key)
    _unique_ratios.append(_ratio)
    _teacher_match_count += _matches
    _total_completions += len(_action_keys)
    _parseable_count += _parseable_n
    if _i < 4:
        print(f"  row[{_i}] task={_row.get('task_id','?'):20s} "
              f"unique={_unique}/{len(_action_keys)}  "
              f"teacher={_t_key}  matched={_matches}/{len(_action_keys)}  "
              f"parseable={_parseable_n}/{len(_action_keys)}")
        for _j, _k in enumerate(_action_keys):
            print(f"      gen[{_j}] -> {_k}")

_mean_unique_ratio = sum(_unique_ratios) / max(len(_unique_ratios), 1)
_teacher_match_rate = _teacher_match_count / max(_total_completions, 1)
_parse_rate = _parseable_count / max(_total_completions, 1)
print(f"\n  mean unique-action ratio : {_mean_unique_ratio:.2f}  (pass >= 0.40)")
print(f"  agent==teacher rate      : {_teacher_match_rate:.2f}  (pass <= 0.70)")
print(f"  parseable completions    : {_parse_rate:.2f}  (pass >= 0.80)")

_b_ok = (_mean_unique_ratio >= 0.40
         and _teacher_match_rate <= 0.70
         and _parse_rate >= 0.80)
if _b_ok:
    print("  ✅ Cell B PASS — model has enough diversity for GRPO to learn.")
else:
    print("  ⚠️  Cell B WARN — diversity / teacher-overlap below threshold.")
    if _mean_unique_ratio < 0.40:
        print("     → completions are nearly identical; raise temperature or top_p.")
    if _teacher_match_rate > 0.70:
        print("     → SFT over-fit teacher; reduce SFT epochs or rely on the")
        print("       exploration bonus in cascade_grpo_reward (train_grpo.py).")
    if _parse_rate < 0.80:
        print("     → many completions don't parse; check format prompt + max_new_tokens.")
    print("  (Continuing anyway — telemetry inside cascade_grpo_reward will surface")
    print("   per-step variance during training.)")

del _probe_rows, _unique_ratios

print(f"\nStarting GRPO training: {GRPO_STEPS} steps × {GENERATIONS} generations ...")
trainer.train()

# ── Save checkpoint ────────────────────────────────────────────────────────────
save_path = os.path.join(OUTPUT_DIR, 'final')
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"\n✅ GRPO checkpoint saved → {save_path}")

if torch.cuda.is_available():
    print(f"   Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable params: 1,081,344 / 316,200,832 (0.34%)
Starting GRPO training: 150 steps × 4 generations ...
[T4 cast] 192 trainable params re-cast → fp32 after GRPOTrainer init
Starting GRPO training: 150 steps × 4 generations ...


Step,Training Loss
1,0.000000
2,-0.000000
3,0.000000
4,-0.000000
5,0.000000


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 🧪 Cell C — Post-training validation (did GRPO actually move the policy?)
# ══════════════════════════════════════════════════════════════════════════════
# Roll the GRPO-trained model on each task and compare its mean episode reward
# against the heuristic baseline. We don't have a clean SFT-only checkpoint
# loaded in-memory at this point, but the heuristic is the relevant ceiling
# (it's what the teacher was) and a coin-flip random-legal policy is the floor.
#
# Pass criterion: trained model >= heuristic on at least 3/6 tasks AND
# trained model >= random_legal on all 6 tasks. Anything below random means
# GRPO destabilized the policy — investigate before declaring success.
# ══════════════════════════════════════════════════════════════════════════════
import time as _time_c, statistics as _stats_c, random as _rng_c

print("\n" + "═" * 72)
print(" Cell C  ·  Post-training rollout: trained model vs heuristic vs random")
print("═" * 72)

_VAL_SEEDS = [101, 202, 303, 404, 505]   # 5 deterministic eval seeds per task
model.eval()

def _roll_policy(_policy_fn, _task, _seeds):
    """Roll a callable policy(obs, legal_actions) on a task; return mean episode reward."""
    _ep_rewards = []
    for _s in _seeds:
        _env = CascadeEnvironment()
        _obs = _env.reset(task_id=_task, seed=int(_s),
                          scenario_split='val', training_mode=False)
        _total = 0.0
        for _step in range(int(TASK_CONFIGS[_task]['max_steps'])):
            try:
                _legal = _env.get_legal_actions()
            except Exception:
                _legal = []
            try:
                _act = _policy_fn(_obs, _legal)
            except TypeError:
                _act = _policy_fn(_obs)
            except Exception:
                _act = CascadeAction(action_type='wait', target_node_id=None)
            _obs = _env.step(_act)
            _total += float(getattr(_obs, 'reward', 0.0) or 0.0)
            if getattr(_obs, 'done', False):
                break
        _ep_rewards.append(_total)
    return _stats_c.mean(_ep_rewards), _stats_c.pstdev(_ep_rewards) if len(_ep_rewards) > 1 else 0.0

def _trained_model_policy(_obs, _legal):
    _prompt = make_training_prompt_fn(_obs, _legal)
    _enc = tokenizer(_prompt, return_tensors='pt',
                     truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        _gen = model.generate(
            **_enc,
            max_new_tokens       = MAX_COMP_LEN,
            do_sample            = False,    # deterministic at eval
            num_return_sequences = 1,
            pad_token_id         = tokenizer.eos_token_id,
        )
    _txt = tokenizer.decode(
        _gen[0][_enc['input_ids'].shape[1]:], skip_special_tokens=True,
    )
    try:
        return parse_action_from_response(_txt, _obs)
    except Exception:
        return CascadeAction(action_type='wait', target_node_id=None)

def _random_legal_policy(_obs, _legal):
    if not _legal:
        return CascadeAction(action_type='wait', target_node_id=None)
    _l = _rng_c.choice(_legal)
    return CascadeAction(
        action_type    = _l.get('action_type', 'wait'),
        target_node_id = _l.get('target_node_id'),
        parameters     = _l.get('parameters') or {},
    )

_heuristic = train_grpo.make_heuristic_policy(CascadeAction)
_results = []
for _task in TASKS:
    print(f"\n  task: {_task}")
    _t0 = _time_c.perf_counter()
    _trained_mu, _trained_sd = _roll_policy(_trained_model_policy, _task, _VAL_SEEDS)
    print(f"    trained:    {_trained_mu:+7.3f} ± {_trained_sd:5.3f}   ({_time_c.perf_counter()-_t0:5.1f}s)")
    _t0 = _time_c.perf_counter()
    _heur_mu, _heur_sd = _roll_policy(_heuristic, _task, _VAL_SEEDS)
    print(f"    heuristic:  {_heur_mu:+7.3f} ± {_heur_sd:5.3f}   ({_time_c.perf_counter()-_t0:5.1f}s)")
    _t0 = _time_c.perf_counter()
    _rnd_mu, _rnd_sd = _roll_policy(_random_legal_policy, _task, _VAL_SEEDS)
    print(f"    random:     {_rnd_mu:+7.3f} ± {_rnd_sd:5.3f}   ({_time_c.perf_counter()-_t0:5.1f}s)")
    _results.append({
        'task': _task,
        'trained': _trained_mu, 'heuristic': _heur_mu, 'random': _rnd_mu,
        'beats_heuristic': _trained_mu >= _heur_mu - 1e-3,
        'beats_random':    _trained_mu >= _rnd_mu  - 1e-3,
    })

print("\n" + "─" * 72)
print(" Summary")
print("─" * 72)
print(f"  {'task':25s}  {'trained':>9s}  {'heuristic':>9s}  {'random':>8s}  vs.heur  vs.rand")
for _r in _results:
    print(f"  {_r['task']:25s}  {_r['trained']:+9.3f}  {_r['heuristic']:+9.3f}  "
          f"{_r['random']:+8.3f}     "
          f"{'✓' if _r['beats_heuristic'] else '✗':^7s} "
          f"{'✓' if _r['beats_random'] else '✗':^7s}")

_n_beats_heur = sum(1 for r in _results if r['beats_heuristic'])
_n_beats_rand = sum(1 for r in _results if r['beats_random'])
print(f"\n  Trained beats heuristic on {_n_beats_heur}/{len(_results)} tasks (target >= 3)")
print(f"  Trained beats random    on {_n_beats_rand}/{len(_results)} tasks (target == {len(_results)})")

if _n_beats_heur >= 3 and _n_beats_rand == len(_results):
    print("  ✅ Cell C PASS — GRPO produced a meaningful policy improvement.")
elif _n_beats_rand == len(_results):
    print("  ⚠️  Cell C PARTIAL — beats random everywhere but doesn't yet exceed")
    print("      the heuristic. Consider more GRPO steps, lower LR, or richer prompts.")
else:
    print("  ❌ Cell C FAIL — trained model is below random on at least one task.")
    print("      GRPO has destabilized the policy; investigate reward telemetry")
    print("      (look for ZERO-VARIANCE GROUP / ALL completions matched teacher)")
    print("      and the diversity output of Cell B before re-running.")

model.train()


## 📈 Step 11 — Plot training curves and save as PNG

Merges SFT loss records + GRPO loss + reward records into two publication-quality plots.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np, json, math, os
from pathlib import Path

# ── Merge all log records ──────────────────────────────────────────────────────
# SFT records (offset step to 0)
sft_records = list(sft_logger.records)

# GRPO records (offset step after SFT)
sft_offset = SFT_STEPS
grpo_records_raw = list(grpo_logger.records)
for entry in trainer.state.log_history:
    step = entry.get('step') or entry.get('global_step')
    if step is None:
        continue
    if any(r['step'] == int(step) + sft_offset for r in grpo_records_raw):
        continue
    rec = {'step': int(step) + sft_offset}
    for key, out_key in [
        ('loss','loss'),('train/loss','loss'),
        ('reward','reward'),('train/reward','reward'),
        ('rewards','reward'),('mean_reward','reward'),
        ('grad_norm','grad_norm'),('learning_rate','lr'),
    ]:
        if key in entry and out_key not in rec:
            v = entry[key]
            rec[out_key] = float(v) if v is not None else float('nan')
    if len(rec) > 1:
        grpo_records_raw.append(rec)

grpo_records_raw.sort(key=lambda r: r['step'])
all_records = sorted(sft_records + grpo_records_raw, key=lambda r: r['step'])
print(f"Total log records : {len(all_records)}  (SFT: {len(sft_records)}, GRPO: {len(grpo_records_raw)})")

def extract(key, records=None):
    records = records or all_records
    xs = [r['step'] for r in records if key in r and not math.isnan(r[key])]
    ys = [r[key]    for r in records if key in r and not math.isnan(r[key])]
    return np.array(xs), np.array(ys)

def smooth(ys, w=None):
    w = w or max(3, len(ys)//10)
    return np.convolve(ys, np.ones(w)/w, mode='valid'), w

def savefig(fig, name):
    p1 = os.path.join(CURVES_DIR, name)
    p2 = os.path.join(OUTPUT_DIR, name)
    fig.savefig(p1, dpi=160, bbox_inches='tight')
    fig.savefig(p2, dpi=160, bbox_inches='tight')
    print(f"Saved: {p1}")
    return p1

saved_files = []

# ─────────────────────────────────────────────────────────────────────────────
# LOSS CURVE (SFT + GRPO combined)
# ─────────────────────────────────────────────────────────────────────────────
xs_l, ys_l = extract('loss')
fig, ax = plt.subplots(figsize=(10, 4.5))

if len(xs_l) > 0:
    # Colour SFT and GRPO segments differently
    sft_mask  = xs_l <= SFT_STEPS
    grpo_mask = xs_l  > SFT_STEPS
    if sft_mask.any():
        ax.plot(xs_l[sft_mask], ys_l[sft_mask],
                color='#3a7ebf', linewidth=1.0, alpha=0.55, label='SFT loss')
    if grpo_mask.any():
        ax.plot(xs_l[grpo_mask], ys_l[grpo_mask],
                color='#e05c5c', linewidth=1.0, alpha=0.55, label='GRPO loss')
    ys_sm, w = smooth(ys_l)
    xs_sm = xs_l[w-1:]
    ax.plot(xs_sm, ys_sm, color='#1a1a1a', linewidth=2.0, label=f'Smoothed (w={w})')
    ax.axvline(SFT_STEPS, color='#888', linestyle=':', linewidth=1.2, label='SFT → GRPO')
    ax.set_ylim(bottom=max(0, ys_l.min()*0.9))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))
    ax.annotate(f'Final: {ys_l[-1]:.4f}',
                xy=(xs_l[-1], ys_l[-1]), xytext=(-60, 12),
                textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='#333'),
                fontsize=9, color='#333')
else:
    ax.text(0.5, 0.5, 'No loss data in log', ha='center', va='center',
            transform=ax.transAxes, fontsize=12, color='gray')

ax.set_title('CascadeGuard — Training Loss (SFT + GRPO)', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('Training step', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25, linestyle='--')
plt.tight_layout()
saved_files.append(savefig(fig, 'training_loss_curve.png'))
plt.show(); plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# REWARD CURVE (GRPO only)
# ─────────────────────────────────────────────────────────────────────────────
xs_r, ys_r = extract('reward', grpo_records_raw)
fig, ax = plt.subplots(figsize=(10, 4.5))

if len(xs_r) > 0:
    ax.plot(xs_r, ys_r, color='#3a7ebf', linewidth=1.0, alpha=0.55, label='Mean reward (per step)')
    ys_sm, w = smooth(ys_r)
    xs_sm = xs_r[w-1:]
    ax.plot(xs_sm, ys_sm, color='#003070', linewidth=2.5, label=f'Smoothed (w={w})')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', label='Baseline = 0')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))
    ax.annotate(f'Final: {ys_r[-1]:.4f}',
                xy=(xs_r[-1], ys_r[-1]), xytext=(-60, 12),
                textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='#003070'),
                fontsize=9, color='#003070')
else:
    ax.text(0.5, 0.5, 'No reward data in log\n(check trainer logs)',
            ha='center', va='center', transform=ax.transAxes, fontsize=12, color='gray')

ax.set_title('CascadeGuard GRPO — Training Reward (normalised)', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('Training step', fontsize=11)
ax.set_ylabel('Mean group reward (GRPO-normalised)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25, linestyle='--')
plt.tight_layout()
saved_files.append(savefig(fig, 'training_reward_curve.png'))
plt.show(); plt.close()

# ── Save raw log JSON ──────────────────────────────────────────────────────────
log_path = os.path.join(CURVES_DIR, 'training_log.json')
with open(log_path, 'w') as f:
    json.dump(all_records, f, indent=2, default=str)
print(f"Raw log  : {log_path}")
print('\n✅ Both curves saved.')

## 🆚 Step 12 — Trained agent vs heuristic baseline (quantitative)

Runs both policies on the same validation seeds and produces a side-by-side comparison.

In [ ]:
if not args.no_eval:
    import pandas as pd, matplotlib, time
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    print("Evaluating heuristic baseline ...")
    t0 = time.perf_counter()
    heuristic_df = train_grpo.evaluate_policy(
        'heuristic', heuristic_policy, args.eval_tasks, args.eval_seeds,
        CascadeEnvironment, TASK_CONFIGS, TASK_SEED_SPLITS,
    )
    print(f"  Done in {time.perf_counter()-t0:.0f}s")

    print("Evaluating trained policy ...")
    t0 = time.perf_counter()
    import torch

    def trained_policy(obs, env=None):
        prompt = make_training_prompt_fn(
            obs, env=env, grpo_mode=True, action_only=args.action_only_prompts,
        )
        inputs = tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN,
        ).to(model.device)
        import time as _t
        t_start = _t.perf_counter()
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_COMP_LEN,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed_ms = (_t.perf_counter() - t_start) * 1000
        new_toks = out[0][inputs['input_ids'].shape[1]:]
        text = tokenizer.decode(new_toks, skip_special_tokens=True)
        parse_failed = not (hasattr(train_grpo, '_looks_parseable_action') and
                            train_grpo._looks_parseable_action(text))
        return parse_action_from_response(text, obs), elapsed_ms, {'parse_failed': parse_failed}

    trained_df = train_grpo.evaluate_policy(
        'grpo_trained', trained_policy, args.eval_tasks, args.eval_seeds,
        CascadeEnvironment, TASK_CONFIGS, TASK_SEED_SPLITS,
    )
    print(f"  Done in {time.perf_counter()-t0:.0f}s")

    # ── Summary ────────────────────────────────────────────────────────────────
    all_df = pd.concat([heuristic_df, trained_df], ignore_index=True)
    summary = (
        all_df.groupby('policy')
        .agg(
            episodes            = ('score', 'count'),
            mean_score          = ('score', 'mean'),
            success_rate        = ('success', 'mean'),
            mean_reward         = ('mean_reward', 'mean'),
            invalid_action_rate = ('invalid_action_rate', 'mean'),
            parse_fail_rate     = ('parse_failure_rate', 'mean'),
            mean_budget_spent   = ('budget_spent', 'mean'),
            avg_resp_ms         = ('avg_response_ms', 'mean'),
        )
        .reset_index()
    )
    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.max_columns', None)
    print('\n=== Policy Comparison ===')
    print(summary.to_string(index=False))

    task_summary = (
        all_df.groupby(['policy', 'task_id'])
        .agg(mean_score=('score','mean'), success_rate=('success','mean'))
        .reset_index()
    )
    print('\n=== Per-Task Breakdown ===')
    print(task_summary.to_string(index=False))

    all_df.to_csv(RESULTS_CSV, index=False)
    summary.to_csv(f'{REPO_DIR}/cascadeguard_eval_summary.csv', index=False)
    print(f'\nResults saved → {RESULTS_CSV}')

    # ── Bar chart ──────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    policies = summary['policy'].tolist()
    scores   = summary['mean_score'].tolist()
    colors   = ['#aab4c4' if 'heuristic' in p else '#e05c5c' for p in policies]

    axes[0].bar(policies, scores, color=colors, edgecolor='white', linewidth=0.8)
    for i, (p, s) in enumerate(zip(policies, scores)):
        axes[0].text(i, s+0.005, f'{s:.3f}', ha='center', va='bottom',
                     fontsize=10, fontweight='bold')
    axes[0].set_title('Mean Episode Score\n(higher is better)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Policy', fontsize=10)
    axes[0].set_ylabel('Mean score (0–1)', fontsize=10)
    axes[0].set_ylim(0, 1.0)
    axes[0].grid(True, axis='y', alpha=0.3, linestyle='--')

    success = summary['success_rate'].tolist()
    axes[1].bar(policies, success, color=colors, edgecolor='white', linewidth=0.8)
    for i, (p, s) in enumerate(zip(policies, success)):
        axes[1].text(i, s+0.005, f'{s:.1%}', ha='center', va='bottom',
                     fontsize=10, fontweight='bold')
    axes[1].set_title('Episode Success Rate\n(higher is better)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Policy', fontsize=10)
    axes[1].set_ylabel('Success rate', fontsize=10)
    axes[1].set_ylim(0, 1.0)
    axes[1].grid(True, axis='y', alpha=0.3, linestyle='--')

    plt.suptitle('CascadeGuard: Trained vs Heuristic Baseline',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    comp_path = os.path.join(CURVES_DIR, 'policy_comparison.png')
    plt.savefig(comp_path, dpi=160, bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR, 'policy_comparison.png'), dpi=160, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Comparison chart saved → {comp_path}')
else:
    print("Eval skipped (RUN_EVAL=False).")

## 🚀 Step 13 — Commit training evidence to GitHub

**Required by the validation bot**: `.png` files must be in the repo.  
Fill in your `GITHUB_TOKEN` (classic PAT with `repo` scope) before running.

In [ ]:
import os, subprocess
from pathlib import Path

# ── FILL IN YOUR DETAILS ───────────────────────────────────────────────────────
GITHUB_TOKEN = ''                             # ← paste your PAT here
GIT_EMAIL    = 'samarthdave0305@example.com'  # ← your GitHub email
GIT_NAME     = 'Samarth Dave'
# ──────────────────────────────────────────────────────────────────────────────

if not GITHUB_TOKEN:
    print("⚠️  GITHUB_TOKEN is empty — skipping push.")
    print("   Add your PAT above, then re-run this cell.")
else:
    REPO_DIR = '/content/cascade_gaurd_openEnv'

    def git(*a):
        r = subprocess.run(['git', '-C', REPO_DIR, *a], capture_output=True, text=True)
        out = (r.stdout + r.stderr).strip()
        if out: print(out)
        return r.returncode

    git('config', 'user.email', GIT_EMAIL)
    git('config', 'user.name',  GIT_NAME)
    git('remote', 'set-url', 'origin',
        f'https://{GITHUB_TOKEN}@github.com/Samarth-Dave/cascade_gaurd_openEnv.git')

    git('add', 'training_curves/')
    git('add', 'cascadeguard_eval_results.csv')
    git('add', 'cascadeguard_eval_summary.csv')

    nb = os.path.join(REPO_DIR, 'CascadeGuard_GRPO_Colab.ipynb')
    if Path(nb).exists():
        git('add', 'CascadeGuard_GRPO_Colab.ipynb')

    rc = git('commit', '-m',
             f'training: SFT {SFT_STEPS}steps + GRPO {GRPO_STEPS}steps — curves + eval')
    if rc == 0:
        git('push', 'origin', 'main')
        print('\n✅ Training evidence committed and pushed to GitHub.')
    else:
        print('Nothing new to commit (already up to date).')

## ✅ Submission checklist

| # | Item | File in repo |
|---|------|-------------|
| 1 | Loss curve | `training_curves/training_loss_curve.png` |
| 2 | Reward curve | `training_curves/training_reward_curve.png` |
| 3 | Policy comparison chart | `training_curves/policy_comparison.png` |
| 4 | Eval results CSV | `cascadeguard_eval_results.csv` |
| 5 | Eval summary CSV | `cascadeguard_eval_summary.csv` |
| 6 | Runnable notebook | `CascadeGuard_GRPO_Colab.ipynb` |
| 7 | Training script | `training/train_grpo.py` |
| 8 | HF Space | https://huggingface.co/spaces/samarthdave0305/cascade-failure-env |

> Make sure Steps 10–13 have all run **without errors** before the deadline.